# Training Environment Setup

Checks that your training environment is ready and exports the variables that every module and workshop uses.

## Learning objectives

- Name your own catalog, its schemas and the `datasets` Volume
- Use the variables that `%run ../setup/00_setup` gives every notebook (`CATALOG`, `DATASET_PATH`, ...)
- Know what to do when one of the checks fails

> **Tip:** The workshops come with the **Workshop Guide** app. The trainer shares its link; you also find it under **Compute → Apps**.

## Before you start

- The trainer has run `00_pre_config` and added you to the training group.
- Self-study / Databricks Free Edition: run `00_pre_config` yourself first, with `SINGLE_USER_MODE = True`.
- Every module and workshop runs this notebook with `%run ../setup/00_setup`.

> **Note:** Do not modify this notebook, except `TRAINER_ACCOUNTS` (trainer's own copy, must match `00_pre_config`) and the `*_OVERRIDE` values.
> It is the single source of truth for environment flags.

## 1. Configuration

Constants shared by all notebooks. `PROGRESS_TABLE` is the shared table where the Workshop Guide app records your results (in self-study, the `_checks` notebooks write it).

In [ ]:
# Overrides for the trainer: None = detect automatically, True / False = force
import os
import re

IS_SERVERLESS_OVERRIDE = None
SINGLE_USER_MODE_OVERRIDE = None

# IS_SERVERLESS: True on serverless compute (the only compute in Databricks Free Edition).
# Heuristic, not a documented contract: serverless notebooks report DATABRICKS_RUNTIME_VERSION
# as "client.<version>", classic clusters as "<major>.<minor>". If it ever guesses wrong,
# set IS_SERVERLESS_OVERRIDE. Demos of features serverless does not support check this flag.
if IS_SERVERLESS_OVERRIDE is None:
    IS_SERVERLESS = os.environ.get("DATABRICKS_RUNTIME_VERSION", "").startswith("client.")
else:
    IS_SERVERLESS = IS_SERVERLESS_OVERRIDE

In [ ]:
# Names of the catalog, schemas, Volume and shared objects
CATALOG_PREFIX = "retailhub"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
DEFAULT_SCHEMA = "default"
VOLUME_NAME = "datasets"

# Account group for the M06 GRANT/REVOKE demo -- created once by the trainer
# (Settings -> Identity and access -> Groups), checked by 00_pre_config.
TRAINING_ANALYSTS_GROUP = "retailhub_analysts"

# 00_pre_config writes this marker into the catalog comment when it runs with SINGLE_USER_MODE = True
SINGLE_USER_MARKER = "[single-user]"

# Workshop progress table, written by the Workshop Guide app; self-study: by check() in _checks (must match 00_pre_config)
PROGRESS_TABLE = "retailhub_trainer.default.workshop_progress"

In [ ]:
# --- BEGIN shared block: keep identical in 00_setup and 00_pre_config ---
# Trainer accounts share one catalog, retailhub_trainer.
# List exact e-mail addresses (or logins before "@"). A "trainer" substring is NOT enough.
TRAINER_ACCOUNTS = []  # e.g. ["trainer@your-company.com"]


def user_slug_for(email):
    """E-mail -> catalog suffix. Example: Jan.Kowalski@example.com -> jan_kowalski."""
    email = email.strip().lower()
    login = email.split("@")[0]
    trainers = {account.strip().lower() for account in TRAINER_ACCOUNTS}
    if email in trainers or login in trainers:
        return "trainer"
    slug = re.sub(r"[^a-z0-9]", "_", login)
    slug = re.sub(r"_+", "_", slug)
    slug = re.sub(r"^[0-9_]+", "", slug)  # a catalog name must not start with a digit
    return slug.strip("_") or "user"
# --- END shared block ---

## 2. Who you are

In [ ]:
# Get the current user from the Databricks session
raw_user = spark.sql("SELECT current_user()").first()[0]
print(f"User: {raw_user}")

# Catalog suffix (e-mail -> slug), same rule as in 00_pre_config
user_slug = user_slug_for(raw_user)
print(f"Slug: {user_slug}")

## 3. Your catalog

Your catalog is `retailhub_<your login>`. The trainer created it in `00_pre_config`.

In [ ]:
# Find your catalog and make it the current one
CATALOG = f"{CATALOG_PREFIX}_{user_slug}"

catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]

if CATALOG in catalogs:
    print(f"[OK] Catalog found: {CATALOG}")
    spark.sql(f"USE CATALOG {CATALOG}")
else:
    print(f"[ERROR] Catalog '{CATALOG}' does not exist!")
    print(f"\nAvailable catalogs with prefix '{CATALOG_PREFIX}':")
    for c in catalogs:
        if c.startswith(CATALOG_PREFIX):
            print(f"  - {c}")
    print("\nContact the trainer -- they need to run 00_pre_config.ipynb")
    print("Self-study / Free Edition: run 00_pre_config.ipynb yourself (SINGLE_USER_MODE = True)")
    raise Exception(f"Catalog '{CATALOG}' not found")

## 4. Environment mode

`SINGLE_USER_MODE` is `True` only when `00_pre_config` ran in single-user mode (self-study, Databricks Free Edition).
It decides which group the M06 GRANT demo uses: the trainer's account group in a classroom, the built-in `account users` group otherwise.

In [ ]:
# Read the environment mode from the catalog comment set by 00_pre_config
if SINGLE_USER_MODE_OVERRIDE is None:
    catalog_info = spark.sql(f"DESCRIBE CATALOG EXTENDED {CATALOG}").collect()
    comments = [row[1] or "" for row in catalog_info if row[0] == "Comment"]
    SINGLE_USER_MODE = any(SINGLE_USER_MARKER in c for c in comments)
else:
    SINGLE_USER_MODE = SINGLE_USER_MODE_OVERRIDE

# Classroom: the trainer's account group. Single user: nobody else to share with,
# so M06 grants to the built-in group of all account users.
ANALYSTS_GROUP = "account users" if SINGLE_USER_MODE else TRAINING_ANALYSTS_GROUP

print(f"[OK] Mode: {'single-user' if SINGLE_USER_MODE else 'classroom'} | M06 demo group: {ANALYSTS_GROUP}")

## 5. Medallion schemas

Your catalog needs the `bronze`, `silver` and `gold` schemas, plus `default` for the Volume.

In [ ]:
# Verify the bronze, silver, gold and default schemas
schema_names = [row[0] for row in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()]
required_schemas = [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA, DEFAULT_SCHEMA]
missing = [s for s in required_schemas if s not in schema_names]

if missing:
    print(f"[ERROR] Missing schemas: {missing}")
    print("Contact the trainer.")
    raise Exception(f"Missing schemas: {missing}")
else:
    print(f"[OK] All schemas present: {', '.join(required_schemas)}")

## 6. Dataset Volume

The training files live in the `datasets` Volume: `/Volumes/<your catalog>/default/datasets`.

In [ ]:
# Verify the Volume and list its folders
DATASET_PATH = f"/Volumes/{CATALOG}/{DEFAULT_SCHEMA}/{VOLUME_NAME}"

try:
    files = dbutils.fs.ls(DATASET_PATH)
    print(f"[OK] Volume accessible: {DATASET_PATH}")
    print(f"Contents:")
    for f in files:
        print(f"   {f.name}")
except Exception as e:
    print(f"[ERROR] Cannot access Volume: {DATASET_PATH}")
    print(f"   Error: {e}")
    raise

## 7. Exported variables

After `%run ../setup/00_setup` every notebook has `CATALOG`, `BRONZE_SCHEMA`, `SILVER_SCHEMA`, `GOLD_SCHEMA`,
`DATASET_PATH`, `IS_SERVERLESS`, `SINGLE_USER_MODE`, `ANALYSTS_GROUP` and `PROGRESS_TABLE`.

In [ ]:
# Print the exported variables
catalog = CATALOG  # aliases for convenience
volume_path = DATASET_PATH

print("=" * 60)
print("TRAINING ENVIRONMENT READY")
print("=" * 60)
print(f"  User:             {raw_user}")
print(f"  CATALOG:          {CATALOG}")
print(f"  BRONZE_SCHEMA:    {BRONZE_SCHEMA}")
print(f"  SILVER_SCHEMA:    {SILVER_SCHEMA}")
print(f"  GOLD_SCHEMA:      {GOLD_SCHEMA}")
print(f"  DATASET_PATH:     {DATASET_PATH}")
print(f"  IS_SERVERLESS:    {IS_SERVERLESS}")
print(f"  SINGLE_USER_MODE: {SINGLE_USER_MODE}")
print(f"  ANALYSTS_GROUP:   {ANALYSTS_GROUP}")
print(f"  PROGRESS_TABLE:   {PROGRESS_TABLE}")
print("=" * 60)

## Summary

- Your work goes to your own catalog `retailhub_<your login>` with the `bronze`, `silver`, `gold` and `default` schemas.
- The training files are in the Volume at `DATASET_PATH`.
- Every notebook gets these variables with `%run ../setup/00_setup`; an `[ERROR]` line tells you what to ask the trainer about.

**Next:** [01_platform_workspace](../modules/01_platform_workspace.ipynb).